# Common
Notebook version of `Service/common.py`.

In [ ]:
from pathlib import Path
from typing import Any, Sequence

import joblib
import numpy as np
from scipy import signal, stats

CURRENT_DIR = Path.cwd()
BACKEND_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "Service_jupyter" else CURRENT_DIR
TRAINED_MODEL_FILE = BACKEND_DIR / "Algorithm" / "saved_model" / "trained_model.pkl"

SAMPLING_RATE = 200
WINDOW_SECONDS = 4.0
FILTER_LOW_HZ = 1.0
FILTER_HIGH_HZ = 45.0
FILTER_ORDER = 4
SELECTED_SEED_CHANNELS = ["TP7", "TP8", "AF3", "AF4"]
LABEL_NAMES = {0: "neutral", 1: "sad", 2: "fear", 3: "happy"}
EEG_BANDS = {
    "delta": (1.0, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta": (13.0, 30.0),
    "gamma": (30.0, 45.0),
}
EPS = 1e-12

def load_model(model_file: Path = TRAINED_MODEL_FILE) -> dict[str, Any]:
    if not model_file.exists():
        raise FileNotFoundError(f"Trained model not found at {model_file}. Run train_model.py first.")
    artifact = joblib.load(model_file)
    if "model" not in artifact:
        raise ValueError("Invalid model artifact: missing 'model'.")
    return artifact

def integrate_power(values: np.ndarray, frequencies: np.ndarray) -> float:
    if values.size < 2:
        return 0.0
    return float(np.trapezoid(values, frequencies))

def compute_band_powers(channel_signal: np.ndarray, sampling_rate: int = SAMPLING_RATE, bands=EEG_BANDS):
    segment_length = min(len(channel_signal), 2 * sampling_rate)
    frequencies, psd = signal.welch(channel_signal, fs=sampling_rate, nperseg=segment_length)
    powers = {}
    for band_name, (low, high) in bands.items():
        mask = (frequencies >= low) & (frequencies < high)
        powers[band_name] = integrate_power(psd[mask], frequencies[mask]) if mask.any() else 0.0
    return powers

def hjorth_parameters(channel_signal: np.ndarray):
    x = np.asarray(channel_signal, dtype=np.float64)
    dx = np.diff(x)
    ddx = np.diff(dx)
    activity = float(np.var(x))
    var_dx = float(np.var(dx))
    var_ddx = float(np.var(ddx))
    mobility = np.sqrt(var_dx / (activity + EPS))
    mobility_dx = np.sqrt(var_ddx / (var_dx + EPS))
    complexity = mobility_dx / (mobility + EPS)
    return activity, float(mobility), float(complexity)

def differential_entropy(variance: float) -> float:
    return float(0.5 * np.log(2.0 * np.pi * np.e * max(variance, EPS)))

def statistical_features(channel_signal: np.ndarray):
    x = np.asarray(channel_signal, dtype=np.float64)
    centered = x - np.mean(x)
    zero_crossings = np.count_nonzero(np.diff(np.signbit(centered)))
    return {
        "mean": float(np.mean(x)), "variance": float(np.var(x)), "std": float(np.std(x)),
        "median": float(np.median(x)), "min": float(np.min(x)), "max": float(np.max(x)),
        "ptp": float(np.ptp(x)), "skewness": float(stats.skew(x, bias=False, nan_policy="omit")),
        "kurtosis": float(stats.kurtosis(x, bias=False, nan_policy="omit")),
        "rms": float(np.sqrt(np.mean(np.square(x)))),
        "zero_crossing_rate": float(zero_crossings / max(1, len(x) - 1)),
    }

def extract_channel_features(channel_signal: np.ndarray, channel_name: str, sampling_rate: int = SAMPLING_RATE):
    values, names = [], []
    for feature_name, value in statistical_features(channel_signal).items():
        names.append(f"{channel_name}_{feature_name}"); values.append(value)
    activity, mobility, complexity = hjorth_parameters(channel_signal)
    extra = {
        "hjorth_activity": activity,
        "hjorth_mobility": mobility,
        "hjorth_complexity": complexity,
        "differential_entropy": differential_entropy(activity),
    }
    for feature_name, value in extra.items():
        names.append(f"{channel_name}_{feature_name}"); values.append(value)
    band_powers = compute_band_powers(channel_signal, sampling_rate=sampling_rate)
    total_power = sum(band_powers.values()) + EPS
    names.append(f"{channel_name}_total_band_power"); values.append(float(total_power))
    for band_name, power in band_powers.items():
        band_features = {
            "power": float(power),
            "relative_power": float(power / total_power),
            "log_power": float(np.log(power + EPS)),
            "differential_entropy": differential_entropy(power),
        }
        for feature_name, value in band_features.items():
            names.append(f"{channel_name}_{band_name}_{feature_name}"); values.append(value)
    return values, names

def extract_feature_vector(window: np.ndarray, sampling_rate: int = SAMPLING_RATE, channel_names: Sequence[str] | None = None):
    eeg_window = np.asarray(window, dtype=np.float64)
    if eeg_window.ndim != 2:
        raise ValueError(f"Expected a 2D channels x samples window, got {eeg_window.shape}")
    channel_names = channel_names or SELECTED_SEED_CHANNELS
    if len(channel_names) != eeg_window.shape[0]:
        raise ValueError(f"Expected {len(channel_names)} channels, got {eeg_window.shape[0]}")
    values, names = [], []
    for i, channel_name in enumerate(channel_names):
        channel_values, channel_names_out = extract_channel_features(eeg_window[i], channel_name, sampling_rate)
        values.extend(channel_values); names.extend(channel_names_out)
    return np.asarray(values, dtype=np.float32), names

print(f"Common helpers loaded. Model path: {TRAINED_MODEL_FILE}")